# Stage 2 — Data Validation & Image Processing (Person 2)

Validate image integrity and save standardized 224x224 RGB images.

**Input:** Stage 1 `outputs/cleaned_dataset.csv`

**Outputs:**
- `outputs/processed_images/`
- `outputs/validated_dataset.csv`
- `outputs/image_validation_report.md`

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from pipeline.stage1_cleaning.cleaning_logic import find_repo_root
from pipeline.stage2_validation.validation_logic import run_image_validation
from pipeline.utils.io import ensure_output_dir, load_csv, save_csv
from pipeline.utils.reports import distribution_markdown, write_markdown_report

REPO_ROOT = find_repo_root(NOTEBOOK_DIR)
INPUT_CSV = REPO_ROOT / 'pipeline' / 'stage1_cleaning' / 'outputs' / 'cleaned_dataset.csv'
OUTPUT_DIR = REPO_ROOT / 'pipeline' / 'stage2_validation' / 'outputs'
PROCESSED_IMAGES_DIR = OUTPUT_DIR / 'processed_images'
ensure_output_dir(OUTPUT_DIR)
ensure_output_dir(PROCESSED_IMAGES_DIR)
print('Input:', INPUT_CSV)
print('Processed images dir:', PROCESSED_IMAGES_DIR)

In [ ]:
df_clean = load_csv(INPUT_CSV)
print('Rows loaded:', len(df_clean))
display(df_clean.head())

In [ ]:
df_valid, validation_stats = run_image_validation(df_clean, PROCESSED_IMAGES_DIR)
print('Corrupted/unreadable:', validation_stats['corrupted_or_unreadable'])
print('Converted to RGB:', validation_stats['converted_to_rgb'])
print('Resized to 224x224:', validation_stats['resized_to_224'])
print('Rows kept:', validation_stats['output_count'])
display(pd.Series(validation_stats['final_class_distribution'], name='count'))

In [ ]:
validated_csv = OUTPUT_DIR / 'validated_dataset.csv'
report_md = OUTPUT_DIR / 'image_validation_report.md'
save_csv(df_valid, validated_csv)

sections = {
    'Overview': 'Stage 2 validates images, converts to RGB, and resizes to 224x224.',
    'Input rows': f"**{validation_stats['input_count']}**",
    'Corrupted or unreadable images removed': f"**{validation_stats['corrupted_or_unreadable']}**",
    'Converted to RGB': f"**{validation_stats['converted_to_rgb']}**",
    'Resized to 224x224': f"**{validation_stats['resized_to_224']}**",
    'Final validated rows': f"**{validation_stats['output_count']}**",
    'Class distribution': distribution_markdown(
        validation_stats['final_class_distribution']
    ),
    'Processed image layout': '\n'.join(
        f'- `{path}`' for path in validation_stats['output_structure'].values()
    ),
    'Output files': '\n'.join([
        f'- `{validated_csv}`',
        f'- `{PROCESSED_IMAGES_DIR}`',
    ]),
}
write_markdown_report(report_md, sections)
print('Saved:', validated_csv)
print('Saved:', report_md)